# 🛡️ ModernBERT URL Phishing Detector

Welcome to the interactive Google Colab demo for the **ModernBERT URL Phishing Detector**.
Developed by **Ilkay ONAY** & **Bayram BAYRAKTAR**.

This notebook will download the fine-tuned model directly from Hugging Face and allow you to test any URL instantly without needing any local setup.

### 1. Install Dependencies
First, we need to install the PyTorch and Transformers libraries to run the inference.

In [ ]:
!pip install -q transformers torch

### 2. Load the Model & Tokenizer from Hugging Face
The model weights and tokenizer will be downloaded securely from the Hugging Face Hub.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ---------------------------------------------------------
MODEL_ID = "ilkayO/modernbert-phishing-detector"
# ---------------------------------------------------------

print(f"Downloading and loading model '{MODEL_ID}'...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)

print("✅ Model loaded successfully!")

### 3. Define the Inference Engine
This function takes a raw URL string, tokenizes it (max 256 length), and passes it through the ModernBERT model to predict the probability of it being malicious.

In [ ]:
def analyze_url(url):
    # Tokenize the input URL
    inputs = tokenizer(url, return_tensors="pt", truncation=True, max_length=256, padding=True)
    
    # Perform inference without calculating gradients
    with torch.no_grad():
        outputs = model(**inputs)
        # Apply softmax to get percentages (0.0 to 1.0)
        probs = F.softmax(outputs.logits, dim=-1)
        
    # Extract probabilities for Safe (0) and Phishing (1) classes
    safe_prob = probs[0][0].item() * 100
    phish_prob = probs[0][1].item() * 100
    
    # Determine Risk Level Thresholds
    if phish_prob > 85:
        risk = "🔴 HIGH RISK (PHISHING)"
    elif phish_prob > 60:
        risk = "🟠 MEDIUM RISK (SUSPICIOUS)"
    else:
        risk = "🟢 LOW RISK (SAFE)"
        
    # Print the detailed report
    print(f"🔗 URL: {url}")
    print(f"🛡️ Verdict: {risk}")
    print(f"📊 Confidence: {phish_prob:.2f}% Malicious | {safe_prob:.2f}% Safe")
    print("-" * 60)

### 4. Test the AI
Let's test the model with a mix of safe and well-known phishing/malicious URL patterns.

In [ ]:
# List of URLs to test
test_urls = [
    "https://www.github.com/ilkay-onay",                      # Safe
    "https://secure-login-verify.suspicious-domain.tk/login", # Phishing
    "http://paypal-verify-account.xyz/secure/login.php",      # Phishing
    "https://www.google.com/search?q=machine+learning",       # Safe
    "http://free-iphone-winner.click/claim-now"               # Phishing
]

# Run the analysis
print("\n🚀 STARTING BATCH ANALYSIS...\n" + "="*60)
for u in test_urls:
    analyze_url(u)